<a href="https://colab.research.google.com/github/Batmansanty49/ProcesamientoDigitalDeSe-alesProyecto1/blob/main/Proyecto_FFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 1
Integrantes:



*   Hernandez Nativitas Sofia Alejandra
*   Ortiz Briseño Emiliano
*   Ruiz Ramírez Santiago



In [1]:
import math
import random
#Numero maximo de muestras
N_MAX = 1024

In [2]:
#Verificamos si N es compatible con el algoritmo de la mariposa
def es_potencia_de_2(n):
    return n >= 2 and (n & (n - 1)) == 0

#Lista de W a utilizar, tomando como base el circulo unitario dividido en N
def calcular_W(N):
    #Paso de 2*pi/N
    paso = 2 * math.pi / N
    #Lista vacia
    W = []

  #Buscamos la mitad del rango, ya que la otra mitad son de misma magnitud y diferente signo
    for k in range(N // 2):
        #El angulo será k * tamaño de nuestra division
        ang = k * paso
        #Usamos complex para que cree un numero complejo usando la formula: cos(ang) − j·sin(ang)
        W.append(complex(math.cos(ang), -math.sin(ang)))
    return W

#Invierte los bits para saber cuales x[n] seran las entradas del algoritmo
#Usamos como entrada el numero de bits que tenemos y el numero
def invertir_bits(i, num_bits):
    #Inicializamos en 0
    resultado = 0
    #Usamos _ para no crear una variable, ya que solo queremos iterar
    #Tomamos el num de bits del numero para las iteraciones
    for _ in range(num_bits):
        #Desplazamos resultado a la izquierda para dejar un espacio libre a la derecha
        #Aplicamos un and de 1 con el ultimo bit de i (nuestro numero), lo que deja solo al bit de la derecha
        #Aplicamos un or entre ambos para que se inserte en resultado
        resultado = (resultado << 1) | (i & 1)
        #Desplazamos en una unidad a la derecha a nuestro numero i para encontrar el siguiente bit
        i >>= 1
    return resultado

#Recibimos como entrada la señal
def fft(x):
    #Leemos el tamaño (N)
    N = len(x)
    #Si no es potencia de 2 lanzamos un error
    if not es_potencia_de_2(N):
        raise ValueError("N debe ser potencia de 2")
    #Si N es mayor a 1024 lanzamos un error
    if N > N_MAX:
        raise ValueError(f"N máximo permitido: {N_MAX}")

    #Leemos el numero de bits que debemos de tener para poder representar a N
    #Por ejemplo, si necesitamos 8 bits, debemos de considerar de 0 a 7, por eso restamos 1
    num_bits = N.bit_length() - 1

    #Calculamos nuestra lista W, donde vendran los indices que provienen del circulo unitario
    W = calcular_W(N)

    #Lista de entrada
    #Inicializamos la entrada con un arreglo de 0's de tamaño N
    X = [0] * N
    #Recorremos el arreglo, invirtiendo los indices y reasignandolos
    for i in range(N):
    #Ejemplo con N=8
    #x[0] = x[0]
    #x[7] = x[1]
    #x[2] = x[2]
    #x[6] = x[3]
        X[invertir_bits(i, num_bits)] = x[i]

    #Etapas de mariposas
    #Las etapas siempre inician con tamaño 2, por eso las inicializamos así
    #y va cambiando cada ciclo, incrementandose *2. tam 2,4,8...
    tam = 2
    #El tamaño solo puede ser menor o igual a N, lo que indica si ya se tomo toda la señal
    while tam <= N:
        #Dividimos a la mitad el tamaño de la muestra
        #debemos de tener dos mitades en cada bloque para hacer la mariposa
        #la distancia entre los dos elementos de una mariposa es mitad del bloque
        #tam 2, distancia 1; tam 4, distancia 2; tam 8, distancia 4
        mitad = tam // 2
        #Esto nos sirve para encontrar el paso que daremos en W
        salto_W = N // tam # índice de W a usar: W_tam^k = W_N^(k*N/tam)
        #Inicio sirve como un multiplicador para el tamaño, permitiendonos pasar las etapas
        for inicio in range(0, N, tam):
            #k sirve para controlar la mariposa dentro de los bloques
            for k in range(mitad):
                #k * salto_W calcula el W que usaremos
                w = W[k * salto_W]
                #a es el elemento de la primer mitad que no se multiplica
                a = X[inicio + k]
                #b es el elemento de la primer mitad que sí se multiplica
                b = X[inicio + k + mitad] * w
                # Mariposa
                #Toma cada posicion de x[n] y combina ambos elementos y su conjugacion
                X[inicio + k] = a + b
                X[inicio + k + mitad] = a - b
        tam *= 2

    return X

In [3]:
def dft_directa(x):
    """DFT O(N^2) solo para verificar resultados."""
    N = len(x)
    out = []
    for k in range(N):
        s = 0j
        for n in range(N):
            ang = 2 * math.pi * k * n / N
            s += x[n] * complex(math.cos(ang), -math.sin(ang))
        out.append(s)
    return out

if __name__ == "__main__":

    print(f"Ejercicio 1")

    #Prueba para N=4
    x= [-4, -4, -12, -16]
    print(f"\nx={x}")

    x_fft_4 = fft(x)

    #fft x[n]
    print("\nFFT:")
    for k, v in enumerate(x_fft_4):
      print(f"  X[{k}] = {v.real:.4f} {v.imag:+.4f}j")

    x_fft_real_4 = dft_directa(x)
    #error = max(abs(u - v) for u, v in zip(a, b))
    error = max(abs(u - v) for u, v in zip(x_fft_4, x_fft_real_4))
    print(f"\nN={4}  error máximo vs DFT: {error:.2e}")

    print(f"\nEjercicio 2")

    #Prueba para N=8
    x= [-1, 2, 0, 0, 4, 0, -3, -1]
    x_fft = fft(x)
    print(f"\nx={x}")

    #fft x[n]
    print("\nFFT:")
    for k, v in enumerate(x_fft):
      print(f"  X[{k}] = {v.real:.4f} {v.imag:+.4f}j")

    x_fft_real = dft_directa(x)
    #error = max(abs(u - v) for u, v in zip(a, b))
    error = max(abs(u - v) for u, v in zip(x_fft, x_fft_real))
    print(f"\nN={8}  error máximo vs DFT: {error:.2e}")

    print(f"\nPruebas hasta 1024")
    for N in (16, 64, 256, 1024):
        x = [random.uniform(-1, 1) for _ in range(N)]
        a, b = fft(x), dft_directa(x)
        error = max(abs(u - v) for u, v in zip(a, b))
        print(f"N={N:3d}  error máximo vs DFT: {error:.2e}")



Ejercicio 1

x=[-4, -4, -12, -16]

FFT:
  X[0] = -36.0000 +0.0000j
  X[1] = 8.0000 -12.0000j
  X[2] = 4.0000 +0.0000j
  X[3] = 8.0000 +12.0000j

N=4  error máximo vs DFT: 8.75e-15

Ejercicio 2

x=[-1, 2, 0, 0, 4, 0, -3, -1]

FFT:
  X[0] = 1.0000 +0.0000j
  X[1] = -4.2929 -5.1213j
  X[2] = 6.0000 -3.0000j
  X[3] = -5.7071 +0.8787j
  X[4] = -1.0000 +0.0000j
  X[5] = -5.7071 -0.8787j
  X[6] = 6.0000 +3.0000j
  X[7] = -4.2929 +5.1213j

N=8  error máximo vs DFT: 6.30e-15

Pruebas hasta 1024
N= 16  error máximo vs DFT: 1.92e-14
N= 64  error máximo vs DFT: 1.24e-13
N=256  error máximo vs DFT: 2.09e-12
N=1024  error máximo vs DFT: 1.59e-11
